# For RFDiffusion2

* Method 1 (Apptainer): https://github.com/RosettaCommons/RFdiffusion2?tab=readme-ov-file
* Method 2 (Source): https://rosettacommons.github.io/RFdiffusion2/installation.html

In [1]:
!git clone https://github.com/RosettaCommons/RFdiffusion2.git ../Tools/RFdiffusion2

Cloning into '../Tools/RFdiffusion2'...
remote: Enumerating objects: 5263, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 5263 (delta 58), reused 55 (delta 35), pack-reused 5167 (from 2)
Receiving objects: 100% (5263/5263), 147.43 MiB | 10.62 MiB/s, done.
Resolving deltas: 100% (2355/2355), done.
Updating files: 100% (5604/5604), done.


## Method 1

In [ ]:
!export PYTHONPATH=Tools/RFdiffusion2
!python Tools/RFdiffusion2/setup.py

## Method 2

In [ ]:
!conda env create -f Tools/RFdiffusion2/envs/cuda124_env.yml
!conda activate rfd2_env_124

In [ ]:
%pip install -r Tools/RFdiffusion2/envs/requirements_cuda124.txt #if necessary

Remove contents from FILES variable from `Tools/RFdiffusion2/setup.py` (the .sif files are not needed for manual setup)

In [ ]:

!cd Tools/Tools/RFdiffusion2
!python setup.py

For newer graphics cards, you may need:

In [ ]:
!conda uninstall pytorch -y
!pip uninstall torch -y

!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Activate conda environment

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(device='cuda'))
print(torch.cuda.get_device_properties(device='cuda'))

True
NVIDIA GeForce RTX 5070 Ti
_CudaDeviceProperties(name='NVIDIA GeForce RTX 5070 Ti', major=12, minor=0, total_memory=16302MB, multi_processor_count=70)


## Running Inference (Source Installation)

Since we are running from source, we point directly to the `pipeline.py` script in the repository.

In [3]:
import os
import sys
import subprocess

# Add RFdiffusion2 to path for imports
REPO_ROOT = os.path.abspath("../Tools/RFdiffusion2")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

try:
    import rf_diffusion
    print(f"Successfully imported rf_diffusion from {os.path.dirname(rf_diffusion.__file__)}")
except ImportError as e:
    print(f"Failed to import rf_diffusion: {e}")

Successfully imported rf_diffusion from /home/ryangustafson/Documents/GitHubProj/PhD-Research/Tools/RFdiffusion2/rf_diffusion


In [4]:
PIPELINE_SCRIPT = os.path.join(REPO_ROOT, "rf_diffusion", "benchmark", "pipeline.py")

def run_inference(config_name, overrides, work_dir=None):
    """
    Helper to run RFdiffusion2 pipeline from valid python environment.
    """
    cmd = [sys.executable, PIPELINE_SCRIPT, f"--config-name={config_name}"] + overrides
    
    # in_proc=True is often needed for local runs without SLURM
    if not any("in_proc" in s for s in overrides):
        cmd.append("in_proc=True")

    print(f"Running: {' '.join(cmd)}")
    
    try:
        subprocess.run(cmd, cwd=work_dir, check=True)
        print("Inference completed successfully.")
    except subprocess.CalledProcessError as e:
        print(f"Inference failed with exit code {e.returncode}")

In [ ]:
# Example: Run a demo case
# This uses the open_source_demo config and a specific benchmark sweep case.

demo_overrides = [
    "sweep.benchmarks=active_site_unindexed_atomic_partial_ligand"
]

run_inference("open_source_demo", demo_overrides)

## Analysis

Load and analyze results using the `rf_diffusion` package or standard pandas.

In [ ]:
# Example analysis loading
import pandas as pd
from rf_diffusion.dev import benchmark as bm

# You would point this to your actual output directory
output_dir = "pipeline_outputs/2025-02-06_14-00-00_open_source_demo"
if os.path.exists(output_dir):
    metrics_df = bm.load_metrics(output_dir)
    print(metrics_df.head())